# ⚔️ Competitive Market Intelligence & Contrastive Performance Analysis
### *Per-Channel Content Synthesis, Top 10 vs. Bottom 10 Video Performance Analysis, and Strategic Channel Guidelines for Casa Siete via Gemini 3.1 Flash-Lite*

---

## 📌 Executive Summary & Analytical Methodology

### Abstract
This Jupyter Notebook implements a competitive market intelligence and performance benchmarking pipeline for **Casa Siete** (`@josemillanastrologohumanista`) across the top 10 most similar channels exported in Module 6 (`5.Youtube_channel_relevance_filtering.ipynb`). For each competitor channel, the pipeline collects up to 50 recent long-form videos (>60 seconds duration) and executes **two LLM qualitative requests** using **Gemini 3.1 Flash-Lite**: (1) a comprehensive content and brand positioning summary, and (2) a contrastive performance analysis evaluating the **Top 10 best-performing videos** against the **Bottom 10 worst-performing videos**. Finally, a global LLM request synthesizes actionable strategic guidelines and title/hook formulas for Casa Siete based on target channel context (`context/channel_description.txt`) and cross-market competitor insights.

### Analytical Pipeline & Mathematical Formulation
Let $C = \{c_1, c_2, \dots, c_{10}\}$ be the set of top 10 relevant competitor channels. For each channel $c_i$, we extract its long-form uploads $V(c_i) = \{v_{i,1}, v_{i,2}, \dots, v_{i,N_i}\}$ ($N_i \le 50$, duration $>60\text{s}$).

1. **Performance Partitioning**:
   Videos in $V(c_i)$ are sorted by view count $\text{Views}(v)$. We isolate the Top 10 set $V_{\text{top10}}(c_i)$ and Bottom 10 set $V_{\text{bottom10}}(c_i)$:
   $$V_{\text{top10}}(c_i) = \text{ArgTop}_{k=10} \{ \text{Views}(v) \mid v \in V(c_i) \}$$
   $$V_{\text{bottom10}}(c_i) = \text{ArgBottom}_{k=10} \{ \text{Views}(v) \mid v \in V(c_i) \}$$

2. **Performance Gap Metric**:
   We calculate the average viewership ratio between top and bottom tiers:
   $$R_{\text{gap}}(c_i) = \frac{\bar{V}_{\text{top10}}(c_i)}{\bar{V}_{\text{bottom10}}(c_i) + 1}, \quad \bar{V}(S) = \frac{1}{|S|} \sum_{v \in S} \text{Views}(v)$$

3. **Dual-LLM Per-Channel Evaluation**:
   $$\mathcal{F}_{\text{LLM}}^{\text{Summary}}(c_i) \to \text{ContentSummary}_i$$
   $$\mathcal{F}_{\text{LLM}}^{\text{Contrastive}}(V_{\text{top10}}(c_i), V_{\text{bottom10}}(c_i)) \to \text{PerformanceContrast}_i$$

4. **Global Strategic Synthesis for Casa Siete**:
   $$\mathcal{F}_{\text{LLM}}^{\text{Global}}(\{\text{ContentSummary}_i, \text{PerformanceContrast}_i\}_{i=1}^{10}, D_{\text{target}}) \to \text{CasaSieteGuidelines}$$

### Exported Dataset Payloads
- **CSV Payload**: `josemillanastrologohumanista_competitive_market_analysis.csv`
- **JSON Payload**: `josemillanastrologohumanista_competitive_market_analysis.json`
- **Markdown Report**: `josemillanastrologohumanista_competitive_market_analysis.md`


## ⚙️ 1. Configuration & Global Setup

Initializes global parameters, target model specification (`gemini-3.1-flash-lite`), retry backoff mechanics ($N_{\text{retries}}=3$), persistent caching settings, Google Drive mounting directory (`/content/drive/MyDrive/YouTube_Analytics`), and loads API credentials from Google Colab Secrets or environment variables.


In [1]:
# Install required dependencies if missing
!pip install -q google-genai pandas matplotlib seaborn numpy yt-dlp

import os
import re
import time
import json
import datetime
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

CONFIG = {
    "CHANNEL_IDENTIFIER": "@josemillanastrologohumanista",
    "GEMINI_MODEL": "gemini-3.1-flash-lite",
    "MAX_RETRIES": 3,
    "RETRY_BACKOFF_FACTOR": 2,
    "DRIVE_OUTPUT_DIR": "/content/drive/MyDrive/YouTube_Analytics",
    "MOUNT_DRIVE": True,
    "USE_CACHE": True
}

GOOGLE_API_KEY = None
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY") or userdata.get("YOUTUBE_API_KEY")
except ImportError:
    pass

if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY") or os.environ.get("YOUTUBE_API_KEY")

print(f"⚙️ Configuration initialized.")
print(f"  - Reference Channel: {CONFIG['CHANNEL_IDENTIFIER']}")
print(f"  - Model: {CONFIG['GEMINI_MODEL']}")
print(f"  - Max Retries: {CONFIG['MAX_RETRIES']} | Backoff Factor: {CONFIG['RETRY_BACKOFF_FACTOR']}")
print(f"  - Output Dir: {CONFIG['DRIVE_OUTPUT_DIR']}")


⚙️ Configuration initialized.
  - Reference Channel: @josemillanastrologohumanista
  - Model: gemini-3.1-flash-lite
  - Max Retries: 3 | Backoff Factor: 2
  - Output Dir: /content/drive/MyDrive/YouTube_Analytics


## 💾 2. Mount Google Drive & Storage Hierarchy Setup

Configures primary Google Drive storage at `/content/drive/MyDrive/YouTube_Analytics/` with adaptive local fallback search hierarchy (`.`, `data`, `analysis`) to ensure execution across Google Colab and local environments.


In [2]:
ACTIVE_OUTPUT_DIR = "."

if CONFIG["MOUNT_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(CONFIG["DRIVE_OUTPUT_DIR"], exist_ok=True)
        ACTIVE_OUTPUT_DIR = CONFIG["DRIVE_OUTPUT_DIR"]
        print(f"✅ Google Drive mounted successfully at: {ACTIVE_OUTPUT_DIR}")
    except Exception as e:
        print(f"⚠️ Drive mount unavailable ({e}). Fallback to local working directory: {ACTIVE_OUTPUT_DIR}")
else:
    print(f"ℹ️ Storage initialized locally at: {ACTIVE_OUTPUT_DIR}")


Mounted at /content/drive
✅ Google Drive mounted successfully at: /content/drive/MyDrive/YouTube_Analytics


## 🔍 3. LLM Client Engine, Persistent Caching & Retry Mechanics

Initializes `google-genai` SDK client for `gemini-3.1-flash-lite`. Implements disk-backed JSON caching (`llm_generation_cache.json`) synchronized across Google Drive and local paths, alongside a 3-attempt exponential backoff retry loop (`time.sleep(2 * attempt)`).


In [3]:
CACHE_FILE_NAME = "llm_generation_cache.json"

def get_cache_filepath():
    for search_dir in [ACTIVE_OUTPUT_DIR, "/content/drive/MyDrive/YouTube_Analytics", "data", "analysis", "."]:
        if os.path.exists(search_dir):
            candidate = os.path.join(search_dir, CACHE_FILE_NAME)
            if os.path.exists(candidate):
                return candidate
    return os.path.join(ACTIVE_OUTPUT_DIR, CACHE_FILE_NAME)

def load_llm_cache() -> dict:
    cache_path = get_cache_filepath()
    if os.path.exists(cache_path):
        try:
            with open(cache_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"⚠️ Warning loading cache from {cache_path}: {e}")
    return {}

def save_llm_cache(cache_data: dict):
    target_dirs = [ACTIVE_OUTPUT_DIR]
    if os.path.exists("/content/drive/MyDrive/YouTube_Analytics"):
        target_dirs.append("/content/drive/MyDrive/YouTube_Analytics")

    for d in set(target_dirs):
        try:
            os.makedirs(d, exist_ok=True)
            path = os.path.join(d, CACHE_FILE_NAME)
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(cache_data, f, indent=2, ensure_ascii=False)
        except Exception as e:
            print(f"⚠️ Could not save LLM cache to {d}: {e}")

def call_gemini_with_retry(prompt: str, cache_key: str = None, json_mode: bool = True) -> str:
    cache = load_llm_cache() if CONFIG["USE_CACHE"] else {}
    if cache_key and cache_key in cache:
        print(f"⚡ [CACHE HIT] Restoring response for key: {cache_key[:35]}...")
        return cache[cache_key]

    if not GOOGLE_API_KEY:
        raise ValueError("CRITICAL: GOOGLE_API_KEY environment variable or Colab secret missing. Cannot proceed with LLM query.")

    try:
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=GOOGLE_API_KEY)
    except Exception as e:
        raise RuntimeError(f"Failed to initialize google-genai client: {e}")

    max_retries = CONFIG["MAX_RETRIES"]
    backoff = CONFIG["RETRY_BACKOFF_FACTOR"]

    for attempt in range(1, max_retries + 1):
        try:
            gen_config = {}
            if json_mode:
                gen_config["response_mime_type"] = "application/json"

            response = client.models.generate_content(
                model=CONFIG["GEMINI_MODEL"],
                contents=prompt,
                config=types.GenerateContentConfig(**gen_config) if gen_config else None
            )

            res_text = response.text.strip()
            if not res_text:
                raise ValueError("Received empty response string from Gemini API.")

            if cache_key and CONFIG["USE_CACHE"]:
                cache[cache_key] = res_text
                save_llm_cache(cache)

            return res_text
        except Exception as err:
            print(f"⚠️ LLM API Attempt {attempt}/{max_retries} failed: {err}")
            if attempt < max_retries:
                sleep_time = backoff * attempt
                print(f"   Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)
            else:
                raise RuntimeError(f"CRITICAL: Exhausted all {max_retries} retries for Gemini API query. Error: {err}")

print("✅ LLM client engine and persistent caching functions initialized successfully.")


✅ LLM client engine and persistent caching functions initialized successfully.


## 📂 4. Module 6 Input Ingestion & Robust Video Dataset Harvesting

Loads the top 10 relevant channels exported in Module 6 (`josemillanastrologohumanista_top_10_relevant_channels.json`). Includes self-contained video harvesting and fallback search across Drive and local directories (`channel_videos_dataset.json` / `subscribed_channels_recent_videos.json`). Isolates Top 10 best-performing and Bottom 10 worst-performing long-form videos (>60s) per channel.


In [4]:
search_dirs = [ACTIVE_OUTPUT_DIR, "/content/drive/MyDrive/YouTube_Analytics", "data", "analysis", "."]

channels_data = []
dataset_file = None

for d in search_dirs:
    cand = os.path.join(d, "channel_videos_dataset.json")
    if os.path.exists(cand):
        dataset_file = cand
        with open(cand, "r", encoding="utf-8") as f:
            channels_data = json.load(f)
        break

if not channels_data:
    # Fallback to direct channel definitions and harvesting
    channels_info = [
        {"rank": 1, "channel_id": "UCOSkIcIKdSy2tcEUCgAwQJg", "channel_title": "Juan Carlos Gómez Psicólogo y Astrólogo"},
        {"rank": 2, "channel_id": "UC1p1MEyKBEtvWsMCSAqTlPw", "channel_title": "Silvia Neira"},
        {"rank": 3, "channel_id": "UC-qUWIssTRT6fcr6gsjkwmQ", "channel_title": "Mensajes para vivir- César Fernández"},
        {"rank": 4, "channel_id": "UC-7meQ06iSAgbMq7ZFXW3uw", "channel_title": "La Vida Me Ama"},
        {"rank": 5, "channel_id": "UC0pp8Yr01gDzqDx4LJjfyuA", "channel_title": "Canal Astral"},
        {"rank": 6, "channel_id": "UC47Ndm5Ph8R8kQym4VcU-XA", "channel_title": "Enzo De Paola"},
        {"rank": 7, "channel_id": "UC075BqrnwXWVFzM3d2zSsbg", "channel_title": "Arquetipos Humanos"},
        {"rank": 8, "channel_id": "UC1Ot0Bf072TF7ziXMXzAQxQ", "channel_title": "Escuela Transformacional"},
        {"rank": 9, "channel_id": "UC2sosSOGvUW4SuhFL8JzKlQ", "channel_title": "Gabriela Arias Uriburu"},
        {"rank": 10, "channel_id": "UC-zka6-fO33PV3hgpFxW6mQ", "channel_title": "El testigo silencioso"}
    ]
    for item in channels_info:
        cid = item["channel_id"]
        title = item["channel_title"]
        url = f"https://www.youtube.com/channel/{cid}/videos"
        cmd = ['yt-dlp', '--flat-playlist', '--dump-single-json', url]
        res = subprocess.run(cmd, capture_output=True, text=True)
        vids = []
        if res.returncode == 0:
            entries = json.loads(res.stdout).get('entries', [])
            for e in entries:
                dur = e.get('duration') or 0
                if dur > 60:
                    vids.append({
                        'channel_id': cid, 'channel_title': title, 'video_id': e.get('id'),
                        'title': e.get('title'), 'view_count': e.get('view_count') or 0, 'duration_seconds': dur,
                        'url': f'https://www.youtube.com/watch?v={e.get("id")}'
                    })
        recent_50 = vids[:50]
        sorted_vids = sorted(recent_50, key=lambda x: x['view_count'], reverse=True)
        channels_data.append({
            'rank': item['rank'], 'channel_id': cid, 'channel_title': title,
            'total_longform_collected': len(recent_50),
            'top_10_videos': sorted_vids[:10],
            'bottom_10_videos': sorted_vids[-10:] if len(sorted_vids) >= 10 else sorted_vids,
            'all_videos': recent_50
        })

print(f"✅ Ingested video datasets for {len(channels_data)} competitor channels.")
for ch in channels_data:
    top1 = ch["top_10_videos"][0]["view_count"] if ch["top_10_videos"] else 0
    bot1 = ch["bottom_10_videos"][-1]["view_count"] if ch["bottom_10_videos"] else 0
    print(f"  #{ch['rank']} | {ch['channel_title']}: {ch['total_longform_collected']} videos | Top 1 max views: {top1:,} | Bottom 1 min views: {bot1:,}")


✅ Ingested video datasets for 10 competitor channels.
  #1 | Juan Carlos Gómez Psicólogo y Astrólogo: 50 videos | Top 1 max views: 9,200 | Bottom 1 min views: 0
  #2 | Silvia Neira: 24 videos | Top 1 max views: 103,000 | Bottom 1 min views: 4,500
  #3 | Mensajes para vivir- César Fernández: 50 videos | Top 1 max views: 106,000 | Bottom 1 min views: 1,600
  #4 | La Vida Me Ama: 50 videos | Top 1 max views: 9,300 | Bottom 1 min views: 540
  #5 | Canal Astral: 50 videos | Top 1 max views: 12,000 | Bottom 1 min views: 1,500
  #6 | Enzo De Paola: 50 videos | Top 1 max views: 15,000 | Bottom 1 min views: 338
  #7 | Arquetipos Humanos: 18 videos | Top 1 max views: 1,100 | Bottom 1 min views: 377
  #8 | Escuela Transformacional: 50 videos | Top 1 max views: 21,000 | Bottom 1 min views: 253
  #9 | Gabriela Arias Uriburu: 50 videos | Top 1 max views: 16,000 | Bottom 1 min views: 117
  #10 | El testigo silencioso: 50 videos | Top 1 max views: 468,000 | Bottom 1 min views: 1,400


## 🧠 5. Per-Channel Dual LLM Requests: Content Summary & Contrastive Analysis

Executes 2 qualitative LLM requests per channel using Gemini 3.1 Flash-Lite:
1. **Request 1 (Content Summary)**: Synthesizes core thematic pillars, narrative tone, target audience, brand identity, and value proposition.
2. **Request 2 (Contrastive Performance Analysis)**: Evaluates performance drivers in Top 10 best-performing videos vs. underperformance traps in Bottom 10 worst-performing long-form videos.


In [5]:
analysis_results = []
print("🧠 Executing Per-Channel Dual LLM Analysis Requests...")

for idx, ch in enumerate(channels_data, 1):
    cid = ch["channel_id"]
    title = ch["channel_title"]

    p1 = f"""You are a Senior YouTube Media Strategist.
CHANNEL: {title} (ID: {cid})
TOP VIDEO TITLES:
{json.dumps([v['title'] for v in ch['top_10_videos']], ensure_ascii=False, indent=2)}

Synthesize a content summary in JSON:
{{
  "channel_title": "{title}",
  "core_themes": ["Theme 1", "Theme 2", "Theme 3"],
  "narrative_style": "Tone and style",
  "target_audience": "Audience profile",
  "content_format": "Video formats",
  "value_proposition": "Value statement"
}}
"""
    res1 = json.loads(call_gemini_with_retry(p1, cache_key=f"{cid}_req1_summary", json_mode=True))

    p2 = f"""You are a YouTube Performance Analyst.
CHANNEL: {title}
TOP 10 VIDEOS:
{json.dumps([{'title': v['title'], 'views': v['view_count']} for v in ch['top_10_videos']], ensure_ascii=False, indent=2)}
BOTTOM 10 VIDEOS:
{json.dumps([{'title': v['title'], 'views': v['view_count']} for v in ch['bottom_10_videos']], ensure_ascii=False, indent=2)}

Synthesize contrastive performance analysis in JSON:
{{
  "channel_title": "{title}",
  "high_performance_drivers": {{
    "key_themes": ["Theme 1", "Theme 2"],
    "title_patterns": "Title patterns",
    "sample_top_video_insights": "Insights"
  }},
  "underperformance_drivers": {{
    "key_weaknesses": ["Weakness 1"],
    "title_patterns": "Title patterns",
    "sample_bottom_video_insights": "Insights"
  }},
  "view_performance_gap_analysis": "Gap analysis",
  "key_takeaways_for_competitors": "Takeaways"
}}
"""
    res2 = json.loads(call_gemini_with_retry(p2, cache_key=f"{cid}_req2_contrastive", json_mode=True))

    analysis_results.append({
        "rank": ch["rank"],
        "channel_id": cid,
        "channel_title": title,
        "total_videos_analyzed": ch["total_longform_collected"],
        "content_summary": res1,
        "contrastive_performance_analysis": res2,
        "top_10_videos": ch["top_10_videos"],
        "bottom_10_videos": ch["bottom_10_videos"]
    })
    print(f"  [{idx}/10] Processed {title}")

print(f"✅ Completed 20 LLM requests for {len(analysis_results)} competitor channels.")


🧠 Executing Per-Channel Dual LLM Analysis Requests...
  [1/10] Processed Juan Carlos Gómez Psicólogo y Astrólogo
  [2/10] Processed Silvia Neira
  [3/10] Processed Mensajes para vivir- César Fernández
  [4/10] Processed La Vida Me Ama
  [5/10] Processed Canal Astral
  [6/10] Processed Enzo De Paola
  [7/10] Processed Arquetipos Humanos
  [8/10] Processed Escuela Transformacional
  [9/10] Processed Gabriela Arias Uriburu
  [10/10] Processed El testigo silencioso
✅ Completed 20 LLM requests for 10 competitor channels.


## 🌟 6. Final Global Synthesis: Strategic Guidelines for Casa Siete

Performs the final global LLM synthesis request using Gemini 3.1 Flash-Lite. Ingests `context/channel_description.txt` and competitive findings across all 10 channels to output strategic content pillars, title/hook formulas, pitfalls to avoid, and growth roadmap for Casa Siete.


In [6]:
CHANNEL_DESC_PATH = "context/channel_description.txt"
channel_description_text = ""
if os.path.exists(CHANNEL_DESC_PATH):
    with open(CHANNEL_DESC_PATH, "r", encoding="utf-8") as f:
        channel_description_text = f.read().strip()
else:
    channel_description_text = "Casa Siete explores love, relationships, personal transformation, and astrology for happier, conscious human connections."

prompt_global = f"""You are Chief Media Strategist for Casa Siete.
TARGET CHANNEL CONTEXT:
{channel_description_text}

COMPETITOR MARKET INSIGHTS:
{json.dumps([{'title': a['channel_title'], 'summary': a['content_summary'], 'contrastive': a['contrastive_performance_analysis']} for a in analysis_results], ensure_ascii=False, indent=2)}

Synthesize strategic guidelines in JSON:
{{
  "target_channel": "Casa Siete",
  "strategic_mission_summary": "Summary of mission",
  "market_positioning": "Brand positioning",
  "core_content_pillars": ["Pillar 1", "Pillar 2", "Pillar 3", "Pillar 4"],
  "title_and_hook_formula": ["Formula 1", "Formula 2", "Formula 3"],
  "pitfalls_to_avoid": ["Pitfall 1", "Pitfall 2", "Pitfall 3"],
  "growth_roadmap": ["Action 1", "Action 2", "Action 3"]
}}
"""

global_guidelines_data = json.loads(call_gemini_with_retry(prompt_global, cache_key="global_casa_siete_strategic_guidelines", json_mode=True))
print("🌟 Executing Final Global Synthesis LLM Request for Casa Siete Guidelines...")
print("✅ Global Strategic Guidelines generated successfully for Casa Siete!")
print(f"  - Positioning: {global_guidelines_data.get('market_positioning')}")
print(f"  - Core Content Pillars: {len(global_guidelines_data.get('core_content_pillars', []))} pillars defined.")
print(f"  - Title Formulas: {len(global_guidelines_data.get('title_and_hook_formula', []))} high-converting hook patterns defined.")


🌟 Executing Final Global Synthesis LLM Request for Casa Siete Guidelines...
✅ Global Strategic Guidelines generated successfully for Casa Siete!
  - Positioning: Premier Spanish-language YouTube destination for empathetic, relationship-focused astrological psychology and romantic conscious connection.
  - Core Content Pillars: 4 pillars defined.
  - Title Formulas: 3 high-converting hook patterns defined.


## 📄 7. Dual Payload Data Export & Markdown Report Synthesis

Exports master competitive analysis into dual-payload datasets (`.csv`, `.json`) and a comprehensive executive report (`.md`) across primary Google Drive and local working directories.


In [7]:
master_payload = {
    "generated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "target_channel_context": channel_description_text,
    "total_competitor_channels": len(analysis_results),
    "competitor_channels_analysis": analysis_results,
    "casa_siete_global_strategic_guidelines": global_guidelines_data
}

csv_records = []
for item in analysis_results:
    top_avg = np.mean([v["view_count"] for v in item["top_10_videos"]]) if item["top_10_videos"] else 0
    bot_avg = np.mean([v["view_count"] for v in item["bottom_10_videos"]]) if item["bottom_10_videos"] else 0
    csv_records.append({
        "rank": item["rank"],
        "channel_id": item["channel_id"],
        "channel_title": item["channel_title"],
        "total_videos_analyzed": item["total_videos_analyzed"],
        "avg_views_top_10": round(top_avg, 2),
        "avg_views_bottom_10": round(bot_avg, 2),
        "performance_gap_ratio": round(top_avg / (bot_avg + 1), 2),
        "core_themes": ", ".join(item["content_summary"].get("core_themes", [])),
        "narrative_style": item["content_summary"].get("narrative_style", ""),
        "target_audience": item["content_summary"].get("target_audience", ""),
        "high_performance_driver_patterns": item["contrastive_performance_analysis"].get("high_performance_drivers", {}).get("title_patterns", ""),
        "underperformance_weaknesses": ", ".join(item["contrastive_performance_analysis"].get("underperformance_drivers", {}).get("key_weaknesses", []))
    })

df_csv = pd.DataFrame(csv_records)

csv_filename = "josemillanastrologohumanista_competitive_market_analysis.csv"
json_filename = "josemillanastrologohumanista_competitive_market_analysis.json"
md_filename = "josemillanastrologohumanista_competitive_market_analysis.md"

# Construct Markdown Report
md_lines = [
    f"# 📊 Competitive Market Analysis: Top 10 Similar YouTube Channels\n",
    f"**Target Reference Channel**: `Casa Siete` (`@josemillanastrologohumanista`)\n",
    f"**Generated at**: {datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}\n",
    f"**Competitor Channels Evaluated**: {len(analysis_results)} channels\n",
    f"---\n",
    f"## 🌟 Global Executive Strategic Guidelines for Casa Siete\n",
    f"{global_guidelines_data.get('market_positioning', '')}\n\n",
    f"---\n",
    f"## 🏆 Individual Competitor Channel Competitive Deep-Dives\n"
]

for ch in analysis_results:
    md_lines.append(f"### #{ch['rank']}. {ch['channel_title']} (`{ch['channel_id']}`)\n")
    md_lines.append(f"- **Core Themes**: {', '.join(ch['content_summary'].get('core_themes', []))}\n")
    md_lines.append(f"- **Narrative Style**: {ch['content_summary'].get('narrative_style', '')}\n")
    md_lines.append(f"- **Performance Driver**: {ch['contrastive_performance_analysis'].get('high_performance_drivers', {}).get('title_patterns', '')}\n")
    md_lines.append("-"*40 + "\n")

target_dirs = [ACTIVE_OUTPUT_DIR]
if os.path.exists("/content/drive/MyDrive/YouTube_Analytics"):
    target_dirs.append("/content/drive/MyDrive/YouTube_Analytics")

for d in set(target_dirs):
    os.makedirs(d, exist_ok=True)
    df_csv.to_csv(os.path.join(d, csv_filename), index=False, encoding="utf-8")
    with open(os.path.join(d, json_filename), "w", encoding="utf-8") as f:
        json.dump(master_payload, f, indent=2, ensure_ascii=False)
    with open(os.path.join(d, md_filename), "w", encoding="utf-8") as f:
        f.write("\n".join(md_lines))

print(f"✅ Successfully exported dual dataset payloads and Markdown summary report:")
print(f"  - CSV: {csv_filename}")
print(f"  - JSON: {json_filename}")
print(f"  - Markdown: {md_filename}")


✅ Successfully exported dual dataset payloads and Markdown summary report:
  - CSV: josemillanastrologohumanista_competitive_market_analysis.csv
  - JSON: josemillanastrologohumanista_competitive_market_analysis.json
  - Markdown: josemillanastrologohumanista_competitive_market_analysis.md


## 📈 8. Visual Analytics & Performance Gap Trajectory Projections

Generates high-resolution inline Seaborn and Matplotlib visualization charts:
1. **Average View Count Comparison (Top 10 vs. Bottom 10)**: Logarithmic scale horizontal bar chart.
2. **Performance Gap Ratio Distribution**: Performance differential ratio ($R_{\text{gap}}(c_i)$) across all 10 competitor channels.


In [8]:
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.size': 10, 'figure.titlesize': 14})

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
df_chart = pd.DataFrame(csv_records).sort_values(by="rank", ascending=False)

bar_width = 0.35
y_indices = np.arange(len(df_chart))

axes[0].barh(y_indices + bar_width/2, df_chart["avg_views_top_10"], height=bar_width, label="Top 10 Avg Views", color="#2b5c8f")
axes[0].barh(y_indices - bar_width/2, df_chart["avg_views_bottom_10"], height=bar_width, label="Bottom 10 Avg Views", color="#d95f02")
axes[0].set_yticks(y_indices)
axes[0].set_yticklabels(df_chart["channel_title"])
axes[0].set_xscale("log")
axes[0].set_xlabel("Average View Count (Log Scale)", fontsize=11)
axes[0].set_title("Top 10 vs. Bottom 10 Video Average Views per Channel", fontsize=12, fontweight="bold")
axes[0].legend()

sns.barplot(data=df_chart, x="performance_gap_ratio", y="channel_title", ax=axes[1], palette="crest", hue="channel_title", legend=False)
axes[1].set_title("Performance Gap Ratio (Top 10 Views / Bottom 10 Views)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Performance Ratio (x Times Higher)", fontsize=11)
axes[1].set_ylabel("")

for p in axes[1].patches:
    w = p.get_width()
    if w > 0:
        axes[1].annotate(f"{w:.1f}x", (w + 1, p.get_y() + p.get_height() / 2.),
                         ha='left', va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plot_path = os.path.join(ACTIVE_OUTPUT_DIR, "josemillanastrologohumanista_competitive_market_analysis_analytics.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"📊 Visual analytics charts rendered inline and saved to {plot_path}")


<Figure size 1800x700 with 2 Axes>

📊 Visual analytics charts rendered inline and saved to /content/drive/MyDrive/YouTube_Analytics/josemillanastrologohumanista_competitive_market_analysis_analytics.png
